# Reproduce MA-dependent ONNX evaluation

이 노트북은 **기존에 MA별로 따로 학습해서 저장한 ONNX 8개**를 이용해 선배의 두 evaluation 방식을 재현합니다.

## 1. For each mass (dedicated model)

각 signal mass point에 **그 mass에서 학습한 전용 ONNX**를 적용합니다.

```text
MA18  sample -> ONNX_MA18  -> ROC/AUC
MA25  sample -> ONNX_MA25  -> ROC/AUC
...
MA125 sample -> ONNX_MA125 -> ROC/AUC
```

TTLJ background도 각 ROC를 만들 때 **같은 전용 ONNX**로 평가합니다.

## 2. Average (mean score of all MA-specific models)

각 event에 8개의 ONNX를 전부 적용하고, 8개 BDT score의 산술평균을 하나의 discriminator로 사용합니다.

```text
event -> ONNX18  -> score18
      -> ONNX25  -> score25
      -> ...
      -> ONNX125 -> score125

mean_score = mean(score18, score25, ..., score125)
```

그 `mean_score`를 사용해 MA18 vs TTLJ, MA25 vs TTLJ, ... ROC/AUC를 각각 계산합니다.

> 주의: "Average"가 선배 코드에서 실제로 단순 score 산술평균이었는지는 선배 코드 확인 전에는 100% 확정할 수 없습니다. 이 노트북은 현재 슬라이드의 `Evaluation: Mean Score (all)` 표현에 가장 자연스러운 **mean of model scores** 방식으로 재현합니다.

이전 MA별 training에서 `TARGET_MA` 기반 feature를 사용하지 않았더라도, **학습 sample 자체가 MA별로 달랐기 때문에 각 ONNX는 MA-dependent dedicated model**입니다.


In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import ROOT
import matplotlib.pyplot as plt
import onnxruntime as ort

from sklearn.metrics import roc_curve, roc_auc_score

np.random.seed(42)


In [ ]:
# ============================================================
# 2. Configuration
#
# 모델 이름 패턴만 실제 저장 이름과 맞는지 확인하세요.
# ============================================================
TREE_NAME = "Training_Tree"

BASE_DIR = "/data9/Users/bhoh/SKNanoOutput/AtobbMLTree/2024"
MODEL_DIR = "/data9/Users/eunsu/MachineLearning/models"

TARGET_MHC = 130
MA_VALUES = [18, 25, 36, 50, 70, 90, 110, 125]

# ------------------------------------------------------------
# Existing MA-specific ONNX model names
#
# 기본 가정:
# xgboost_atobb_MHc130_MA18_ttlJ.onnx
# ...
#
# 실제 파일명이 다르면 이 dictionary만 수정하면 됩니다.
# ------------------------------------------------------------
MODEL_NAMES = {
    ma: f"xgboost_atobb_MHc{TARGET_MHC}_MA{ma}_ttlJ"
    for ma in MA_VALUES
}

ONNX_PATHS = {
    ma: f"{MODEL_DIR}/{MODEL_NAMES[ma]}.onnx"
    for ma in MA_VALUES
}

FEATURE_JSON_PATHS = {
    ma: f"{MODEL_DIR}/{MODEL_NAMES[ma]}_features.json"
    for ma in MA_VALUES
}

SCALER_PICKLE_PATHS = {
    ma: f"{MODEL_DIR}/{MODEL_NAMES[ma]}_scaler.pkl"
    for ma in MA_VALUES
}

SCALER_JSON_PATHS = {
    ma: f"{MODEL_DIR}/{MODEL_NAMES[ma]}_scaler.json"
    for ma in MA_VALUES
}

# ------------------------------------------------------------
# Evaluation ROOT samples
# ------------------------------------------------------------
SIG_FILES = {
    ma: f"{BASE_DIR}/TTToHcToWAToBB-MHc{TARGET_MHC}_MA{ma}_SingleLepFilter.root"
    for ma in MA_VALUES
}

BKG_FILE = f"{BASE_DIR}/TTLJ_powheg.root"

# None = full TTLJ evaluation tree
N_BKG_EVAL = None

WEIGHT_COL = "weight_train"

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------
PLOT_DIR = Path("./onnx_eval_MA_dependent_reproduction")
PDF_DIR = PLOT_DIR / "pdf"
OUTPUT_DIR = PLOT_DIR / "outputs"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("===== ONNX models =====")
for ma in MA_VALUES:
    print(
        f"MA={ma:3d}: ONNX={os.path.exists(ONNX_PATHS[ma])}  "
        f"JSON={os.path.exists(FEATURE_JSON_PATHS[ma])}"
    )
    print("       ", ONNX_PATHS[ma])

print("\n===== ROOT samples =====")
for ma in MA_VALUES:
    print(f"MA={ma:3d}: {os.path.exists(SIG_FILES[ma])}  {SIG_FILES[ma]}")
print("TTLJ:", os.path.exists(BKG_FILE), BKG_FILE)


In [ ]:
# ============================================================
# 3. Resolve model files
#
# If your actual filenames differ only slightly, this cell prints
# missing files clearly before anything expensive is done.
# ============================================================
missing_models = []

for ma in MA_VALUES:
    if not os.path.exists(ONNX_PATHS[ma]):
        missing_models.append(ONNX_PATHS[ma])

    if not os.path.exists(FEATURE_JSON_PATHS[ma]):
        missing_models.append(FEATURE_JSON_PATHS[ma])

if missing_models:
    print("Missing model-related files:")
    for p in missing_models:
        print(" ", p)

    raise FileNotFoundError(
        "MA-specific model filename pattern does not match your saved files. "
        "Edit MODEL_NAMES in cell 2."
    )

print("All MA-specific ONNX + feature JSON files found.")


In [ ]:
# ============================================================
# 4. Load metadata for all models
# ============================================================
model_meta = {}

for ma in MA_VALUES:
    with open(FEATURE_JSON_PATHS[ma]) as f:
        meta = json.load(f)

    model_meta[ma] = {
        "features": meta["features"],
        "scaler_type": meta.get("scaler_type", None),
        "raw": meta,
    }

print("Feature counts:")
for ma in MA_VALUES:
    print(
        f"  MA={ma:3d}: "
        f"{len(model_meta[ma]['features'])} features, "
        f"scaler={model_meta[ma]['scaler_type']}"
    )

# Check whether feature order is identical across dedicated models.
ref_features = model_meta[MA_VALUES[0]]["features"]

same_feature_order = all(
    model_meta[ma]["features"] == ref_features
    for ma in MA_VALUES
)

print("\nSame feature order across all models:", same_feature_order)

if not same_feature_order:
    print(
        "NOTE: This is still supported. Each model will use its own "
        "saved feature order during inference."
    )


In [ ]:
# ============================================================
# 5. ROOT helper + derived bb-summary reconstruction
#
# These four variables were reconstructed in the training notebook:
#   bb_mass_min_all
#   bb_mass_max_all
#   bb_mass_at_min_dr
#   bb_dr_min_all
#
# They are NOT direct ROOT branches.
# ============================================================
DERIVED_BB_SUMMARY_FEATURES = [
    "bb_mass_min_all",
    "bb_mass_max_all",
    "bb_mass_at_min_dr",
    "bb_dr_min_all",
]

BB_PAIRS = ["01", "02", "12", "03", "13", "23"]

BB_SOURCE_BRANCHES = [
    "bb_mass_01", "bb_mass_02", "bb_mass_12",
    "bb_mass_03", "bb_mass_13", "bb_mass_23",
    "bb_dr_01", "bb_dr_02", "bb_dr_12",
    "bb_dr_03", "bb_dr_13", "bb_dr_23",
]

def add_bb_summary_features(df):
    out = df.copy()

    mass_cols = [f"bb_mass_{pair}" for pair in BB_PAIRS]
    dr_cols = [f"bb_dr_{pair}" for pair in BB_PAIRS]

    masses = out[mass_cols].to_numpy(dtype=float)
    drs = out[dr_cols].to_numpy(dtype=float)

    valid = (
        np.isfinite(masses)
        & np.isfinite(drs)
        & (masses > 0.0)
        & (drs > 0.0)
    )

    has_valid_pair = np.any(valid, axis=1)

    mass_for_min = np.where(valid, masses, np.inf)
    mass_for_max = np.where(valid, masses, -np.inf)
    dr_for_min = np.where(valid, drs, np.inf)

    out["bb_mass_min_all"] = np.min(mass_for_min, axis=1)
    out["bb_mass_max_all"] = np.max(mass_for_max, axis=1)

    min_dr_idx = np.argmin(dr_for_min, axis=1)
    row_idx = np.arange(len(out))

    out["bb_dr_min_all"] = drs[row_idx, min_dr_idx]
    out["bb_mass_at_min_dr"] = masses[row_idx, min_dr_idx]

    out.loc[
        ~has_valid_pair,
        DERIVED_BB_SUMMARY_FEATURES,
    ] = np.nan

    return out


def check_required_branches(root_path, tree_name, branches):
    f = ROOT.TFile.Open(root_path, "READ")

    if not f or f.IsZombie():
        raise RuntimeError(f"Cannot open ROOT file: {root_path}")

    tree = f.Get(tree_name)

    if not tree:
        f.Close()
        raise RuntimeError(
            f"Cannot find tree '{tree_name}' in {root_path}"
        )

    available = {b.GetName() for b in tree.GetListOfBranches()}
    missing = [x for x in branches if x not in available]

    f.Close()

    if missing:
        raise RuntimeError(
            f"Missing ROOT branches in {root_path}:\n"
            + "\n".join(missing)
        )


def read_root(root_path, tree_name, branches):
    check_required_branches(root_path, tree_name, branches)

    rdf = ROOT.RDataFrame(tree_name, root_path)
    arrays = rdf.AsNumpy(branches)

    return pd.DataFrame({
        key: np.asarray(value)
        for key, value in arrays.items()
    })


In [ ]:
# ============================================================
# 6. Determine the union of ROOT branches needed by all models
# ============================================================
all_features = sorted(set(
    feat
    for ma in MA_VALUES
    for feat in model_meta[ma]["features"]
))

root_features = [
    feat for feat in all_features
    if feat not in DERIVED_BB_SUMMARY_FEATURES
]

# Source branches are needed only if any derived bb summary is used.
needs_bb_summary = any(
    feat in all_features
    for feat in DERIVED_BB_SUMMARY_FEATURES
)

BRANCHES_TO_READ = list(root_features)

if needs_bb_summary:
    BRANCHES_TO_READ += BB_SOURCE_BRANCHES

BRANCHES_TO_READ += [WEIGHT_COL]
BRANCHES_TO_READ = sorted(set(BRANCHES_TO_READ))

print("Number of unique model features:", len(all_features))
print("Number of ROOT branches to read:", len(BRANCHES_TO_READ))
print("Need derived bb summaries:", needs_bb_summary)


In [ ]:
# ============================================================
# 7. Read evaluation samples ONCE
# ============================================================
sig_dfs = {}

for ma, path in SIG_FILES.items():
    print(f"Reading signal MA={ma} ...")

    df = read_root(
        path,
        TREE_NAME,
        BRANCHES_TO_READ,
    )

    if needs_bb_summary:
        df = add_bb_summary_features(df)

    df["mass_point"] = ma
    df["label"] = 1
    sig_dfs[ma] = df

    print("  entries =", len(df))

print("\nReading TTLJ ...")

df_bkg = read_root(
    BKG_FILE,
    TREE_NAME,
    BRANCHES_TO_READ,
)

if needs_bb_summary:
    df_bkg = add_bb_summary_features(df_bkg)

df_bkg["mass_point"] = -1
df_bkg["label"] = 0

if N_BKG_EVAL is not None and len(df_bkg) > N_BKG_EVAL:
    df_bkg = (
        df_bkg
        .sample(n=N_BKG_EVAL, random_state=42)
        .reset_index(drop=True)
    )

print("TTLJ entries =", len(df_bkg))


In [ ]:
# ============================================================
# 8. Cleaning for inference
#
# Same convention used in training:
#   +/-inf -> NaN
#   <= -998 -> NaN
# ============================================================
def clean_for_inference(df):
    out = df.copy()

    for feat in all_features:
        if feat not in out.columns:
            continue

        out[feat] = pd.to_numeric(out[feat], errors="coerce")
        out.loc[np.isinf(out[feat]), feat] = np.nan
        out.loc[out[feat] <= -998, feat] = np.nan

    out[WEIGHT_COL] = pd.to_numeric(
        out[WEIGHT_COL],
        errors="coerce",
    )

    out = out[
        np.isfinite(out[WEIGHT_COL])
    ].copy()

    return out


for ma in MA_VALUES:
    sig_dfs[ma] = clean_for_inference(sig_dfs[ma])

df_bkg = clean_for_inference(df_bkg)

print("After cleaning:")
for ma in MA_VALUES:
    print(f"  MA={ma:3d}: {len(sig_dfs[ma])}")
print("  TTLJ  :", len(df_bkg))


In [ ]:
# ============================================================
# 9. Load ALL MA-specific ONNX sessions + scalers
# ============================================================
models = {}

for ma in MA_VALUES:
    sess = ort.InferenceSession(
        ONNX_PATHS[ma],
        providers=["CPUExecutionProvider"],
    )

    scaler_type = model_meta[ma]["scaler_type"]
    scaler = None
    scaler_json = None

    if scaler_type is not None:
        if os.path.exists(SCALER_PICKLE_PATHS[ma]):
            with open(SCALER_PICKLE_PATHS[ma], "rb") as f:
                scaler = pickle.load(f)

        elif os.path.exists(SCALER_JSON_PATHS[ma]):
            with open(SCALER_JSON_PATHS[ma]) as f:
                scaler_json = json.load(f)

        else:
            raise FileNotFoundError(
                f"Scaler required for MA={ma} but scaler file was not found."
            )

    models[ma] = {
        "session": sess,
        "input_name": sess.get_inputs()[0].name,
        "features": model_meta[ma]["features"],
        "scaler_type": scaler_type,
        "scaler": scaler,
        "scaler_json": scaler_json,
    }

print("Loaded", len(models), "dedicated ONNX models.")


In [ ]:
# ============================================================
# 10. ONNX inference helpers
# ============================================================
def extract_signal_probability(outputs):
    if len(outputs) == 1:
        arr = np.asarray(outputs[0])

        if arr.ndim == 2 and arr.shape[1] >= 2:
            return arr[:, 1].astype(float)

        return arr.reshape(-1).astype(float)

    prob = outputs[-1]

    if isinstance(prob, list):
        return np.asarray(
            [p[1] if 1 in p else p.get("1") for p in prob],
            dtype=float,
        )

    arr = np.asarray(prob)

    if arr.ndim == 2 and arr.shape[1] >= 2:
        return arr[:, 1].astype(float)

    return arr.reshape(-1).astype(float)


def transform_for_model(df, model_ma):
    info = models[model_ma]
    features = info["features"]

    missing = [
        feat for feat in features
        if feat not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"Missing features for model MA={model_ma}: {missing}"
        )

    X = df[features].to_numpy(dtype=np.float32)

    if info["scaler_type"] is None:
        return X

    if info["scaler"] is not None:
        return info["scaler"].transform(X).astype(np.float32)

    scaler_meta = info["scaler_json"]

    if info["scaler_type"] == "standard":
        mean = np.asarray(scaler_meta["mean"], dtype=np.float32)
        scale = np.asarray(scaler_meta["scale"], dtype=np.float32)
        return ((X - mean) / scale).astype(np.float32)

    if info["scaler_type"] == "minmax":
        min_ = np.asarray(scaler_meta["min"], dtype=np.float32)
        scale = np.asarray(scaler_meta["scale"], dtype=np.float32)
        return (X * scale + min_).astype(np.float32)

    raise ValueError(
        f"Unknown scaler type for MA={model_ma}: "
        f"{info['scaler_type']}"
    )


def predict_model(df, model_ma, batch_size=200000):
    info = models[model_ma]
    X = transform_for_model(df, model_ma)
    preds = []

    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))

        outputs = info["session"].run(
            None,
            {info["input_name"]: X[start:stop]},
        )

        preds.append(
            extract_signal_probability(outputs)
        )

    return np.concatenate(preds)


In [ ]:
# ============================================================
# 11. Run ALL models on ALL samples
#
# This creates the score matrix needed for:
#   - dedicated / "For each mass"
#   - average / mean score of all models
# ============================================================
# Background: one score array per model
bkg_scores_by_model = {}

for model_ma in MA_VALUES:
    print(f"TTLJ <- ONNX MA={model_ma}")
    bkg_scores_by_model[model_ma] = predict_model(
        df_bkg,
        model_ma,
    )

# Signal: every signal mass is evaluated by every ONNX model
sig_scores = {}

for sample_ma in MA_VALUES:
    sig_scores[sample_ma] = {}

    for model_ma in MA_VALUES:
        print(
            f"Signal MA={sample_ma} <- ONNX MA={model_ma}"
        )

        sig_scores[sample_ma][model_ma] = predict_model(
            sig_dfs[sample_ma],
            model_ma,
        )

print("Inference matrix completed.")


In [ ]:
# ============================================================
# 12. Construct the TWO discriminators
#
# (A) Dedicated / For each mass
#     sample MA uses matching ONNX MA
#
# (B) Average
#     arithmetic mean of scores from all 8 ONNX models
# ============================================================
# Dedicated background score depends on which MA ROC is being evaluated.
dedicated_sig_score = {
    ma: sig_scores[ma][ma]
    for ma in MA_VALUES
}

dedicated_bkg_score = {
    ma: bkg_scores_by_model[ma]
    for ma in MA_VALUES
}

# Mean score of ALL dedicated models
average_sig_score = {
    sample_ma: np.mean(
        np.vstack([
            sig_scores[sample_ma][model_ma]
            for model_ma in MA_VALUES
        ]),
        axis=0,
    )
    for sample_ma in MA_VALUES
}

average_bkg_score = np.mean(
    np.vstack([
        bkg_scores_by_model[model_ma]
        for model_ma in MA_VALUES
    ]),
    axis=0,
)

print("Score construction complete.")


In [ ]:
# ============================================================
# 13. ROC helper
# ============================================================
def calculate_roc(
    sig_score,
    bkg_score,
    sig_weight,
    bkg_weight,
):
    y_true = np.concatenate([
        np.ones(len(sig_score), dtype=int),
        np.zeros(len(bkg_score), dtype=int),
    ])

    y_score = np.concatenate([
        sig_score,
        bkg_score,
    ])

    weights = np.concatenate([
        np.abs(np.asarray(sig_weight, dtype=float)),
        np.abs(np.asarray(bkg_weight, dtype=float)),
    ])

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_score,
        sample_weight=weights,
    )

    auc_value = roc_auc_score(
        y_true,
        y_score,
        sample_weight=weights,
    )

    return {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "auc": auc_value,
    }


bkg_weight = df_bkg[WEIGHT_COL].to_numpy(dtype=float)

roc_dedicated = {}
roc_average = {}

for ma in MA_VALUES:
    sig_weight = sig_dfs[ma][WEIGHT_COL].to_numpy(dtype=float)

    roc_dedicated[ma] = calculate_roc(
        dedicated_sig_score[ma],
        dedicated_bkg_score[ma],
        sig_weight,
        bkg_weight,
    )

    roc_average[ma] = calculate_roc(
        average_sig_score[ma],
        average_bkg_score,
        sig_weight,
        bkg_weight,
    )

print("ROC calculation complete.")


In [ ]:
# ============================================================
# 14. Summary table
# ============================================================
rows = []

for ma in MA_VALUES:
    rows.append({
        "MA": ma,
        "AUC_average": roc_average[ma]["auc"],
        "AUC_dedicated": roc_dedicated[ma]["auc"],
        "dedicated_minus_average": (
            roc_dedicated[ma]["auc"]
            - roc_average[ma]["auc"]
        ),
        "N_signal": len(sig_dfs[ma]),
        "N_background": len(df_bkg),
    })

result_df = pd.DataFrame(rows)

display(result_df)

print(
    "Mean AUC - Average   =",
    result_df["AUC_average"].mean(),
)
print(
    "Mean AUC - Dedicated =",
    result_df["AUC_dedicated"].mean(),
)


In [ ]:
# ============================================================
# 15. Plot: Average / Mean Score (all models)
#
# Reproduces the left-hand concept in the senior slide.
# ============================================================
plt.figure(figsize=(8, 7))

for ma in MA_VALUES:
    r = roc_average[ma]

    plt.plot(
        r["fpr"],
        r["tpr"],
        linewidth=1.6,
        label=fr"$m_A={ma}$ GeV (AUC={r['auc']:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

plt.xlabel("Background Efficiency")
plt.ylabel("Signal Efficiency")
plt.title("MA-specific ONNX ensemble: mean score (all models)")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(fontsize=9)
plt.tight_layout()

plt.savefig(PLOT_DIR / "roc_average_all_models.png", dpi=180)
plt.savefig(PDF_DIR / "roc_average_all_models.pdf")
plt.show()


In [ ]:
# ============================================================
# 16. Plot: For each mass / Dedicated model
#
# Reproduces the right-hand concept in the senior slide.
# ============================================================
plt.figure(figsize=(8, 7))

for ma in MA_VALUES:
    r = roc_dedicated[ma]

    plt.plot(
        r["fpr"],
        r["tpr"],
        linewidth=1.6,
        label=fr"$m_A={ma}$ GeV (AUC={r['auc']:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

plt.xlabel("Background Efficiency")
plt.ylabel("Signal Efficiency")
plt.title("MA-specific ONNX: dedicated model for each mass")
plt.xlim(0, 1)
plt.ylim(0, 1.02)
plt.legend(fontsize=9)
plt.tight_layout()

plt.savefig(PLOT_DIR / "roc_dedicated_each_mass.png", dpi=180)
plt.savefig(PDF_DIR / "roc_dedicated_each_mass.pdf")
plt.show()


In [ ]:
# ============================================================
# 17. Plot: AUC vs MA, Average vs Dedicated
# ============================================================
plt.figure(figsize=(8, 5))

plt.plot(
    result_df["MA"],
    result_df["AUC_average"],
    marker="o",
    label="Average score of all MA-specific models",
)

plt.plot(
    result_df["MA"],
    result_df["AUC_dedicated"],
    marker="o",
    label="Dedicated model for each MA",
)

plt.xlabel(r"$m_A$ [GeV]")
plt.ylabel("ROC AUC")
plt.title(f"MA-specific ONNX evaluation (MHc={TARGET_MHC} GeV)")

ymin = min(
    result_df["AUC_average"].min(),
    result_df["AUC_dedicated"].min(),
) - 0.03

ymax = max(
    result_df["AUC_average"].max(),
    result_df["AUC_dedicated"].max(),
) + 0.03

plt.ylim(max(0.5, ymin), min(1.0, ymax))
plt.legend()
plt.tight_layout()

plt.savefig(PLOT_DIR / "auc_vs_MA_average_vs_dedicated.png", dpi=180)
plt.savefig(PDF_DIR / "auc_vs_MA_average_vs_dedicated.pdf")
plt.show()


In [ ]:
# ============================================================
# 18. Optional comparison with NEW combined-MA single model
#
# If you already ran evaluate_combined_MA_ONNX.ipynb and have:
#   .../outputs/auc_by_mass.csv
#
# set the path below and this cell adds the third curve.
# ============================================================
COMBINED_AUC_CSV = (
    "./onnx_eval_xgboost_atobb_MHc130_MAcombined_ttlj/"
    "outputs/auc_by_mass.csv"
)

if os.path.exists(COMBINED_AUC_CSV):
    combined_df = pd.read_csv(COMBINED_AUC_CSV)

    compare_df = result_df.merge(
        combined_df[["MA", "AUC"]].rename(
            columns={"AUC": "AUC_combined_single"}
        ),
        on="MA",
        how="left",
    )

    display(compare_df)

    plt.figure(figsize=(8, 5))

    plt.plot(
        compare_df["MA"],
        compare_df["AUC_average"],
        marker="o",
        label="Average of dedicated scores",
    )

    plt.plot(
        compare_df["MA"],
        compare_df["AUC_dedicated"],
        marker="o",
        label="Dedicated model",
    )

    plt.plot(
        compare_df["MA"],
        compare_df["AUC_combined_single"],
        marker="o",
        label="Combined-MA single model",
    )

    plt.xlabel(r"$m_A$ [GeV]")
    plt.ylabel("ROC AUC")
    plt.title(
        f"BDT strategy comparison (MHc={TARGET_MHC} GeV)"
    )
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        PLOT_DIR / "auc_vs_MA_three_strategies.png",
        dpi=180,
    )
    plt.savefig(
        PDF_DIR / "auc_vs_MA_three_strategies.pdf"
    )
    plt.show()

else:
    print(
        "Combined-MA CSV not found. "
        "Skip 3-way comparison:"
    )
    print(COMBINED_AUC_CSV)


In [ ]:
# ============================================================
# 19. Optional diagnostic:
#     how differently do the 8 models score the SAME event?
#
# Useful if Average performs much worse than Dedicated.
# ============================================================
CHECK_SAMPLE_MA = 90

score_matrix = np.vstack([
    sig_scores[CHECK_SAMPLE_MA][model_ma]
    for model_ma in MA_VALUES
]).T

score_matrix_df = pd.DataFrame(
    score_matrix,
    columns=[f"model_MA{ma}" for ma in MA_VALUES],
)

print(f"Signal sample MA={CHECK_SAMPLE_MA}")
display(score_matrix_df.describe().T)

print("\nCorrelation between MA-specific model scores:")
display(score_matrix_df.corr())


In [ ]:
# ============================================================
# 20. Save numerical results
# ============================================================
csv_path = OUTPUT_DIR / "auc_average_vs_dedicated.csv"
result_df.to_csv(csv_path, index=False)

save_dict = {}

for ma in MA_VALUES:
    save_dict[f"average_MA{ma}_fpr"] = roc_average[ma]["fpr"]
    save_dict[f"average_MA{ma}_tpr"] = roc_average[ma]["tpr"]
    save_dict[f"dedicated_MA{ma}_fpr"] = roc_dedicated[ma]["fpr"]
    save_dict[f"dedicated_MA{ma}_tpr"] = roc_dedicated[ma]["tpr"]

npz_path = OUTPUT_DIR / "roc_average_vs_dedicated.npz"
np.savez(npz_path, **save_dict)

print("Saved:", csv_path)
print("Saved:", npz_path)


## 해석 포인트

### `Average`
각 event에 8개의 MA-specific ONNX를 전부 적용한 뒤 **8개 score를 평균**합니다.  
따라서 mass hypothesis 하나에 강하게 의존하지 않는 ensemble discriminator처럼 동작합니다.

### `For each mass`
각 signal mass point에 **그 mass에서 학습한 ONNX**를 적용합니다.  
예를 들어 MA=90 signal과 TTLJ에는 MA90 ONNX를 적용해서 MA90 ROC를 만듭니다.

### 현재 새 combined-MA single model과의 차이

- **Dedicated**: 8개의 training → 8개의 ONNX → 각 MA에 matching model
- **Average**: 8개의 training → 8개의 ONNX → 각 event score 8개를 평균
- **Combined single model**: 8개 MA signal을 합쳐 **한 번 training** → ONNX 1개 → 모든 MA에 동일 model

따라서 최종 비교 plot은 `AUC vs MA`에 이 세 방법을 동시에 올리는 것이 가장 직관적입니다.

> 이 노트북의 `Average` 정의는 선배 슬라이드의 `Evaluation: Mean Score (all)`을 단순 산술평균으로 해석해 재현한 것입니다. 선배 코드가 있으면 average 계산 부분을 확인해 정확히 동일한 정의인지 검증할 수 있습니다.
